In [1]:
# Install necessary libraries
!pip install pandas plotly folium altair

# Import libraries
import pandas as pd
import plotly.express as px
import folium
from folium.plugins import MarkerCluster
import altair as alt


KeyboardInterrupt: 

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# Load the datasets
dataset = pd.read_csv('dataset.csv')
climate_policy_df = pd.read_csv('final_df_with_climate_and_policy.csv')
migrant_incidents_df = pd.read_csv('North_America_Incidents.csv')
cbp_encounters_df = pd.read_csv('U.S.CustomsandBorderProtection(CBP)Encounters ____________ - Sheet1.csv')

# Quick preview
print("Dataset shape:", dataset.shape)
print("Climate & Policy Data shape:", climate_policy_df.shape)
print("North America Incidents shape:", migrant_incidents_df.shape)
print("CBP Encounters shape:", cbp_encounters_df.shape)




In [ ]:
print(migrant_incidents_df.columns.tolist())



In [ ]:
# Try different possible column names
possible_date_cols = ['Date', 'Reported Date', 'Incident Date', 'date', 'Reported_Date']

# Find which one exists
found_date_col = None
for col in possible_date_cols:
    if col in migrant_incidents_df.columns:
        found_date_col = col
        break

if found_date_col:
    migrant_incidents_df[found_date_col] = pd.to_datetime(migrant_incidents_df[found_date_col], errors='coerce')
    migrant_incidents_df['Parsed_Date'] = migrant_incidents_df[found_date_col]
else:
    print("❌ No date column found!")

# Same thing for dropping missing locations
if 'Latitude' in migrant_incidents_df.columns and 'Longitude' in migrant_incidents_df.columns:
    migrant_incidents_df = migrant_incidents_df.dropna(subset=['Latitude', 'Longitude'])
else:
    print("❌ No Latitude/Longitude columns found!")


In [ ]:
# Try different possible column names
possible_date_cols = ['Date', 'Reported Date', 'Incident Date', 'date', 'Reported_Date', 'ReportedDate']

# Find which one exists
found_date_col = None
for col in possible_date_cols:
    if col in migrant_incidents_df.columns:
        found_date_col = col
        break

if found_date_col:
    migrant_incidents_df[found_date_col] = pd.to_datetime(migrant_incidents_df[found_date_col], errors='coerce')
    migrant_incidents_df['Parsed_Date'] = migrant_incidents_df[found_date_col]
else:
    print("❌ No date column found!")

# Same thing for dropping missing locations
if 'Latitude' in migrant_incidents_df.columns and 'Longitude' in migrant_incidents_df.columns:
    migrant_incidents_df = migrant_incidents_df.dropna(subset=['Latitude', 'Longitude'])
else:
    print("❌ No Latitude/Longitude columns found!")

# Assuming 'ReportedDate' or a similar variation is the actual column name
migrant_incidents_df['Date'] = migrant_incidents_df[found_date_col] # Use the found_date_col here
migrant_incidents_df['Parsed_Date']

In [ ]:
# Parse dates
migrant_incidents_df['Date'] = pd.to_datetime(migrant_incidents_df['Date'], errors='coerce')

# Check if 'Policy_Date' or a similar column exists in climate_policy_df
possible_policy_date_cols = ['Policy_Date', 'policy_date', 'Date', 'date']
# Parse dates
migrant_incidents_df['Date'] = pd.to_datetime(migrant_incidents_df['Date'], errors='coerce')

# Check if 'Policy_Date' or a similar column exists in climate_policy_df
possible_policy_date_cols = ['Policy_Date', 'policy_date', 'Date', 'date']
found_policy_date_col = None

for col in possible_policy_date_cols:
    if col in climate_policy_df.columns:
        found_policy_date_col = col
        break

In [ ]:
import plotly.express as px

# Re-aggregate monthly incidents
time_series = migrant_incidents_df.groupby(migrant_incidents_df['Parsed_Date'].dt.to_period('M')).size().reset_index(name='Incidents')
time_series['Parsed_Date'] = time_series['Parsed_Date'].dt.to_timestamp()

# Animated line chart
animated_fig = px.line(
    time_series,
    x='Parsed_Date', y='Incidents',
    title='Animated Timeline of Migrant Incidents',
    labels={'Parsed_Date':'Date', 'Incidents':'Number of Incidents'},
    width=1000, height=500
)

animated_fig.update_layout(
    font=dict(size=14),
    title_font_size=22,
    hovermode="x unified"
)

animated_fig.show()


In [ ]:
# First, split Coordinates into Latitude and Longitude
migrant_incidents_df[['Latitude', 'Longitude']] = migrant_incidents_df['Coordinates'].str.split(',', expand=True)

# Convert to numeric
migrant_incidents_df['Latitude'] = pd.to_numeric(migrant_incidents_df['Latitude'], errors='coerce')
migrant_incidents_df['Longitude'] = pd.to_numeric(migrant_incidents_df['Longitude'], errors='coerce')


In [ ]:
# Check suspicious values
print(migrant_incidents_df[['Latitude', 'Longitude']].describe())


In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(migrant_incidents_df['Longitude'], migrant_incidents_df['Latitude'], alpha=0.5)
plt.title('Corrected Coordinate Distribution')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(True)
plt.show()


In [ ]:
import folium
from folium.plugins import MarkerCluster

# Function to choose marker color based on number of deaths
def get_marker_color(deaths):
    try:
        deaths = int(deaths)
        if deaths <= 1:
            return 'green'
        elif deaths <= 5:
            return 'orange'
        else:
            return 'red'
    except:
        return 'gray'

# 1. Create the base map
cleaned_map = folium.Map(
    location=[migrant_incidents_df['Latitude'].mean(), migrant_incidents_df['Longitude'].mean()],
    zoom_start=5,
    tiles='cartodb positron'
)

# 2. Add MarkerCluster
marker_cluster = MarkerCluster(name="Migrant Incidents").add_to(cleaned_map)

# 3. Add Markers
for idx, row in migrant_incidents_df.iterrows():
    popup_html = f"""
    <strong>Date:</strong> {row.get('Incident Date', 'Unknown')}<br>
    <strong>Deaths:</strong> {row.get('Total Number of Dead and Missing', 'Unknown')}
    """

    marker_color = get_marker_color(row.get('Total Number of Dead and Missing', 0))

    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=folium.Popup(popup_html, max_width=300),
        icon=folium.Icon(color=marker_color, icon='info-sign')
    ).add_to(marker_cluster)

# 4. Create a custom legend (HTML + CSS)
legend_html = """
<div style="
position: fixed;
bottom: 50px; left: 50px; width: 200px; height: 140px;
background-color: white;
border:2px solid grey;
z-index:9999;
font-size:14px;
padding: 10px;
">
<b>Legend: Deaths per Incident</b><br>
<i class="fa fa-map-marker fa-2x" style="color:green"></i> 0–1 deaths<br>
<i class="fa fa-map-marker fa-2x" style="color:orange"></i> 2–5 deaths<br>
<i class="fa fa-map-marker fa-2x" style="color:red"></i> >5 deaths<br>
<i class="fa fa-map-marker fa-2x" style="color:gray"></i> Unknown deaths
</div>
"""
# 5. Add the legend to the map
cleaned_map.get_root().html.add_child(folium.Element(legend_html))

# 6. Show final map
cleaned_map

In [ ]:
print(migrant_incidents_df.columns.tolist())


The migrant_incidents_cluster_map.html visualizes the geographic distribution of migrant deaths and disappearances across North America. Each marker represents a fatal or missing migrant incident, clustered dynamically to show regional concentrations. The map highlights major hotspots of migrant vulnerability, particularly along the U.S.-Mexico border, and offers an interactive, human-centered view of migration crises through space and time.

It is an interactive web map created using Folium, a Python library that serves as a wrapper around Leaflet.js, a popular JavaScript library for interactive maps. This particular map visualizes migrant death incidents across North America. Each incident is represented as a marker placed on a basemap, and to handle the large volume of individual incidents without cluttering the map, these markers are grouped into clusters. This clustering automatically adjusts as users zoom in and out of the map, expanding into individual points when zoomed close and collapsing into summarized clusters when zoomed out.

**Technical Features**

HTML+CSS+JS based map.

Built automatically by Folium in Python.

Clustering makes it easier to handle hundreds of markers without slowing down.

Popups contain basic date, region, and deaths data for each point.

The map is centered approximately on latitude 30.69 and longitude -107.25, which places it near the U.S.Mexico border — an area historically associated with significant migrant movements. It uses a light-themed basemap provided by CARTO (CartoDB positron tiles), giving it a clean and professional background suited for data visualization.

**Significance**

Overall, this visualization transforms otherwise abstract or hidden data about migrant fatalities into an immediate, visceral spatial story. It enables policymakers, researchers, advocates, and the public to engage critically with the realities faced by migrants, helping to ground policy discussions and humanitarian debates in concrete geographic realities.



In [ ]:
import pandas as pd
import plotly.graph_objects as go

# 0. Load datasets
migrant_incidents_df = pd.read_csv('/content/North_America_Incidents.csv')
climate_policy_df = pd.read_csv('/content/US CPB POLICIES.csv')

# 1. STRIP SPACES immediately
migrant_incidents_df.columns = migrant_incidents_df.columns.str.strip()
climate_policy_df.columns = climate_policy_df.columns.str.strip()

# 2. Rename important columns
climate_policy_df = climate_policy_df.rename(columns={
    'Policy Year': 'Year',
    'Policy Title': 'Policy_Name'
})

# 3. Drop rows without 'Policy_Name'
climate_policy_df = climate_policy_df.dropna(subset=['Policy_Name'])

# 4. Check if 'Year' and 'Policy_Name' exist
if not {'Year', 'Policy_Name'}.issubset(climate_policy_df.columns):
    print("❌ Columns found:", climate_policy_df.columns.tolist())
    raise ValueError("Error: 'Year' and/or 'Policy_Name' columns not found.")

# 5. Prepare 'Date' columns
migrant_incidents_df['Incident Date'] = pd.to_datetime(migrant_incidents_df['Incident Date'], errors='coerce')
climate_policy_df['Policy_Date'] = pd.to_datetime(dict(
    year=climate_policy_df['Year'],
    month=1,
    day=1
))

# 6. Aggregate incidents monthly
incident_time_series = migrant_incidents_df.groupby(migrant_incidents_df['Incident Date'].dt.to_period('M')).size().reset_index(name='Incidents')
incident_time_series['Incident Date'] = incident_time_series['Incident Date'].dt.to_timestamp()

# 🎨 Build Policy Colors
policy_colors = {}
for policy in climate_policy_df['Policy_Name']:
    if any(word in policy.lower() for word in ['deferred', 'relief', 'protection', 'child', 'daca', 'asylum']):
        policy_colors[policy] = 'green'
    else:
        policy_colors[policy] = 'red'

# 🏛️ Define Presidential Eras
presidential_periods = [
    {"president": "Obama", "start": "2009-01-20", "end": "2017-01-20", "color": "#d3e5ff"},
    {"president": "Trump", "start": "2017-01-20", "end": "2021-01-20", "color": "#ffd6d6"},
    {"president": "Biden", "start": "2021-01-20", "end": "2025-01-20", "color": "#e2f7d4"}
]

# 7. Create base Figure
fig = go.Figure()

# 8. Add Presidential Shading
for era in presidential_periods:
    fig.add_vrect(
        x0=era['start'], x1=era['end'],
        fillcolor=era['color'],
        opacity=0.3,
        layer="below",
        line_width=0,
        annotation_text=era['president'],
        annotation_position="top left",
        annotation=dict(font_size=16, font_color="black")
    )

# 9. Add Migrant Incidents Line (Empty Start)
fig.add_trace(go.Scatter(
    x=[], y=[],
    mode='lines+markers',
    line=dict(color='crimson', width=3),
    name='Migrant Incidents'
))

# 10. Create Animation Frames (All policies use green/red)
frames = []
for i in range(len(incident_time_series)):
    current_time = incident_time_series.loc[i, 'Incident Date']
    visible_policies = climate_policy_df[climate_policy_df['Policy_Date'] <= current_time]

    frame_shapes = [
        dict(
            type='line',
            x0=row['Policy_Date'],
            x1=row['Policy_Date'],
            y0=0,
            y1=incident_time_series['Incidents'].max() * 1.1,
            line=dict(color=policy_colors[row['Policy_Name']], dash='dash', width=2)
        )
        for _, row in visible_policies.iterrows()
    ]

    frame_annotations = [
        dict(
            x=row['Policy_Date'],
            y=incident_time_series['Incidents'].max() * 1.05,
            text=row['Policy_Name'],
            showarrow=True,
            arrowhead=2,
            font=dict(size=14, color=policy_colors[row['Policy_Name']]),
            bgcolor='white'
        )
        for _, row in visible_policies.iterrows()
    ]

    frame = go.Frame(
        data=[
            go.Scatter(
                x=incident_time_series.loc[:i, 'Incident Date'],
                y=incident_time_series.loc[:i, 'Incidents'],
                mode='lines+markers',
                line=dict(color='crimson', width=3)
            )
        ],
        layout=go.Layout(
            shapes=frame_shapes,
            annotations=frame_annotations
        ),
        name=str(current_time)[:7]
    )
    frames.append(frame)

fig.frames = frames

# 11. Setup Dropdown Filter
filter_buttons = [
    dict(
        label="All Policies",
        method="relayout",
        args=[{"shapes": [], "annotations": []}]
    )
]

# Special color for filtered lines
filtered_line_color = 'royalblue'

for policy_name in climate_policy_df['Policy_Name'].unique():
    single_shape = [
        dict(
            type="line",
            x0=row['Policy_Date'],
            x1=row['Policy_Date'],
            y0=0,
            y1=incident_time_series['Incidents'].max() * 1.1,
            line=dict(color=filtered_line_color, dash='dash', width=3)
        )
        for _, row in climate_policy_df[climate_policy_df['Policy_Name'] == policy_name].iterrows()
    ]

    single_annotation = [
        dict(
            x=row['Policy_Date'],
            y=incident_time_series['Incidents'].max() * 1.05,
            text=row['Policy_Name'],
            showarrow=True,
            arrowhead=2,
            font=dict(size=14, color=filtered_line_color),
            bgcolor='white'
        )
        for _, row in climate_policy_df[climate_policy_df['Policy_Name'] == policy_name].iterrows()
    ]

    filter_buttons.append(
        dict(
            label=policy_name,
            method="relayout",
            args=[{"shapes": single_shape, "annotations": single_annotation}]
        )
    )

# 12. Final Layout
fig.update_layout(
    title='📈 Migrant Incidents Over Time with Filtered Color Highlighted Policies',
    title_font_size=30,
    xaxis_title='Month-Year',
    yaxis_title='Number of Incidents',
    font=dict(size=18),
    hovermode="x unified",
    width=1300,
    height=750,
    xaxis=dict(
        range=[incident_time_series['Incident Date'].min(), incident_time_series['Incident Date'].max()],
        tickformat="%b %Y",
        rangeslider=dict(visible=True)
    ),
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label="▶️ Play",
                    method="animate",
                    args=[None, {"frame": {"duration": 300, "redraw": True}, "fromcurrent": True}]
                ),
                dict(
                    label="⏸️ Pause",
                    method="animate",
                    args=[[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]
                )
            ],
            direction="left",
            type="buttons",
            pad={"r": 10, "t": 87},
            showactive=True,
            x=0.05,
            xanchor="left",
            y=1.2,
            yanchor="top"
        ),
        dict(
            buttons=filter_buttons,
            direction="down",
            showactive=True,
            x=0.85,
            xanchor="right",
            y=1.2,
            yanchor="top"
        )
    ]
)

# 13. Show Final Animated Figure
fig.show()


This dashboard provides an interactive visualization of migrant death and disappearance incidents across North America, organized month-by-month over multiple years. It also intersects important US immigration policy especially as it relates to category 'inadmissible' and 'expulsion' through vhanging regimes. The time scope spans from 2014- 2023. Built using Plotly and exported as a standalone HTML file, the dashboard enables dynamic exploration of temporal patterns in migrant vulnerability. Users can animate the timeline, filter incidents by year, and manually navigate across months using embedded dropdowns, sliders, and play/pause controls. By translating raw incident data into an accessible, animated timeline, the dashboard highlights seasonal spikes, long-term trends, and moments of intensified risk. It serves as a portable, fully offline tool for researchers, policymakers, humanitarian advocates, and educators seeking to visualize the spatial and temporal dimensions of migrant crises. The dashboard's self-contained design allows it to be easily shared and embedded in reports, presentations, and public-facing digital archives.

**Technical Features**

Embedded Plotly.js library (~very large script inside!).

Embedded JSON objects for your figures data, layout, and animation frames.

No external dependencies are required: offline and portable.

**What it is visualizing**:

Time series of migrant incidents, organized by month and year.

Animation frames allow viewers to watch changes over time.

Dropdown menu lets users filter by year (or see "All Years").

Play/Pause buttons let users animate the changes across months.

Slider at the bottom allows manual navigation across dates.

The dashboard helps reveal seasonal patterns, spikes, and trends in migrant incidents over multiple years.

In [ ]:
from IPython.display import display, HTML

# Adjusted Styling for HTML headings and links
display(HTML("""
<style>
h1, h2, h3 {
  font-family: 'Segoe UI', 'Arial', sans-serif;
  font-weight: 600;
  color: #2C3E50;
  text-align: center;
  margin-bottom: 10px;
}

p {
  font-family: 'Segoe UI', 'Arial', sans-serif;
  font-size: 16px;
  color: #555;
  text-align: center;
  margin-bottom: 20px;
}

a {
  font-family: 'Segoe UI', 'Arial', sans-serif;
  font-size: 18px;
  color: #1ABC9C;
  text-decoration: none;
  font-weight: 600;
}

a:hover {
  text-decoration: underline;
  color: #16A085;
}

body {
  background-color: #F9F9F9;
}
</style>
"""))


In [ ]:
# Save your Folium map
dynamic_map.save('/content/migrant_incidents_map.html')

print("✅ Folium map saved successfully as migrant_incidents_map.html")
